### Transform Constructors Data

1. Read bronze_constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id)
4. Rename columns to make them more meaningful (name → constructor_name)
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver_constructors table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/03.Silver-Helper

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.constructors"
silver_table = F"{catalog_name}.{silver_schema}.constructors"


In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_df = (
    spark.read.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
)

### Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_selected_df = (
    constructors_df.select(
        F.col("constructorId"),
        F.col("name"),
        F.col("nationality"),
        F.col("timestamp"),
        F.col("source_file"),
        F.col("batch_id")
    )
)


In [0]:
display(constructors_selected_df)


### Standardise column names using snake_case (constructorId → constructor_id)
### Rename columns to make them more meaningful (name → constructor_name)

In [0]:
constructors_remaned_df = (
    constructors_selected_df
    .withColumnRenamed("constructorId", "constructor_id")
    .withColumnRenamed("name", "constructor_name")
     
)

In [0]:
display(constructors_remaned_df)

### Remove duplicate records

In [0]:
constructors_dinstince_df = (
    constructors_remaned_df
    .dropDuplicates(["constructor_id"])
)

In [0]:
display(constructors_dinstince_df)

### Transform values of column nationality to Title Case

In [0]:
constructors_final_df = (
    constructors_dinstince_df
    .withColumn("nationality", F.initcap(F.col("nationality")))
   )

In [0]:
display(constructors_final_df)

In [0]:
write_to_silver(
    input_df = constructors_final_df,
    target_table = silver_table,
    merge_condition= "t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "timestamp",
        "source_file",
        "batch_id"
    ]
)


In [0]:
display(spark.table(silver_table))